In [1]:
#Load packages and libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import os
from scipy.sparse import coo_matrix

In [2]:
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#Data loading and preprocessing
train_data=pd.read_json('/content/drive/MyDrive/meta_Electronics.jsonl.gz',lines=True,compression="gzip",nrows=20000)
train_data.columns

Index(['main_category', 'title', 'average_rating', 'rating_number', 'features',
       'description', 'price', 'images', 'videos', 'store', 'categories',
       'details', 'parent_asin', 'bought_together', 'subtitle', 'author'],
      dtype='object')

In [5]:
train_data.shape

(20000, 16)

In [6]:
train_data.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],[Teleporter V3 The “Teleporter V3” kit sets a ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Fat Shark,"[Electronics, Television & Video, Video Glasses]","{'Date First Available': 'August 2, 2014', 'Ma...",B00MCW7G9M,NaN,NaN,NaN
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],SIIG,"[Electronics, Television & Video, Accessories,...",{'Product Dimensions': '0.83 x 4.17 x 2.05 inc...,B00YT6XQSE,NaN,NaN,NaN
2,Computers,Digi-Tatoo Decal Skin Compatible With MacBook ...,4.5,246,[WARNING: Please IDENTIFY MODEL NUMBER on the ...,[],19.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'AL 2Sides Video', 'url': 'https://...",Digi-Tatoo,"[Electronics, Computers & Accessories, Laptop ...","{'Brand': 'Digi-Tatoo', 'Color': 'Fresh Marble...",B07SM135LS,NaN,NaN,NaN
3,AMAZON FASHION,NotoCity Compatible with Vivoactive 4 band 22m...,4.5,233,[☛NotoCity 22mm band is designed for Vivoactiv...,[],9.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],NotoCity,"[Electronics, Wearable Technology, Clips, Arm ...","{'Date First Available': 'May 29, 2020', 'Manu...",B089CNGZCW,NaN,NaN,NaN
4,Cell Phones & Accessories,Motorola Droid X Essentials Combo Pack,3.8,64,"[New Droid X Essentials Combo Pack, Exclusive ...",[all Genuine High Quality Motorola Made Access...,14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Verizon,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '11.6 x 6.9 x 3.1 inche...,B004E2Z88O,NaN,NaN,NaN


In [7]:
train_data['images'][0]

[{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg',
  'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg',
  'variant': 'MAIN',
  'hi_res': None}]

In [11]:
train_data.isnull().sum()

,0
main_category,222
title,0
average_rating,0
rating_number,0
features,0
description,0
price,11363
images,0
videos,0
store,108


In [12]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    19778 non-null  object 
 1   title            20000 non-null  object 
 2   average_rating   20000 non-null  float64
 3   rating_number    20000 non-null  int64  
 4   features         20000 non-null  object 
 5   description      20000 non-null  object 
 6   price            8637 non-null   float64
 7   images           20000 non-null  object 
 8   videos           20000 non-null  object 
 9   store            19892 non-null  object 
 10  categories       20000 non-null  object 
 11  details          20000 non-null  object 
 12  parent_asin      20000 non-null  object 
 13  bought_together  0 non-null      float64
 14  subtitle         7 non-null      object 
 15  author           6 non-null      object 
dtypes: float64(3), int64(1), object(12)
memory usage: 2.4+ MB


In [13]:
#handling null values
train_data['main_category']=train_data['main_category'].fillna('Unknown')

In [14]:
train_data['price']=train_data['price'].fillna('Not available')

In [15]:
train_data['store']=train_data['store'].fillna('Unknown')

In [16]:
train_data.columns

Index(['main_category', 'title', 'average_rating', 'rating_number', 'features',
       'description', 'price', 'images', 'videos', 'store', 'categories',
       'details', 'parent_asin', 'bought_together', 'subtitle', 'author'],
      dtype='object')

In [17]:
train_data.drop(columns=['subtitle','bought_together','author'],inplace=True)

In [18]:
train_data.isnull().sum()

,0
main_category,0
title,0
average_rating,0
rating_number,0
features,0
description,0
price,0
images,0
videos,0
store,0


In [19]:
train_data['parent_asin'].duplicated().sum()

np.int64(0)

In [20]:
train_data.shape

(20000, 13)

In [21]:
train_data

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],[Teleporter V3 The “Teleporter V3” kit sets a ...,Not available,[{'thumb': 'https://m.media-amazon.com/images/...,[],Fat Shark,"[Electronics, Television & Video, Video Glasses]","{'Date First Available': 'August 2, 2014', 'Ma...",B00MCW7G9M
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],Not available,[{'thumb': 'https://m.media-amazon.com/images/...,[],SIIG,"[Electronics, Television & Video, Accessories,...",{'Product Dimensions': '0.83 x 4.17 x 2.05 inc...,B00YT6XQSE
2,Computers,Digi-Tatoo Decal Skin Compatible With MacBook ...,4.5,246,[WARNING: Please IDENTIFY MODEL NUMBER on the ...,[],19.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'AL 2Sides Video', 'url': 'https://...",Digi-Tatoo,"[Electronics, Computers & Accessories, Laptop ...","{'Brand': 'Digi-Tatoo', 'Color': 'Fresh Marble...",B07SM135LS
3,AMAZON FASHION,NotoCity Compatible with Vivoactive 4 band 22m...,4.5,233,[☛NotoCity 22mm band is designed for Vivoactiv...,[],9.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],NotoCity,"[Electronics, Wearable Technology, Clips, Arm ...","{'Date First Available': 'May 29, 2020', 'Manu...",B089CNGZCW
4,Cell Phones & Accessories,Motorola Droid X Essentials Combo Pack,3.8,64,"[New Droid X Essentials Combo Pack, Exclusive ...",[all Genuine High Quality Motorola Made Access...,14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Verizon,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '11.6 x 6.9 x 3.1 inche...,B004E2Z88O
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,Amazon Home,Power on Board 500 Watt Dc-to-AC Power Inverter,1.0,1,"[500 Watt with 1000 Watt surge, 2 outlets so y...",[The VEC052POB is a 500 Watt power inverter. T...,Not available,[{'thumb': 'https://m.media-amazon.com/images/...,[],VECTOR,"[Electronics, Car & Vehicle Electronics, Vehic...",{'Product Dimensions': '15.5 x 10.25 x 2.5 inc...,B0006HTQJS
19996,Computers,LB1 High Performance Battery for Toshiba Satel...,2.9,9,[Battery Type: Li-ion; Capacity: 4400mAh/48Wh;...,[About LB1 High PerformanceAt LB1 High Perform...,Not available,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Troubleshooting Battery Video', 'u...",LB1 High Performance,"[Electronics, Computers & Accessories, Laptop ...","{'Brand': 'LB1 High Performance', 'Item model ...",B008N1LCN2
19997,Computers,RackPath 1U Blank Rack Mount Panel Spacer (10 ...,4.8,7,"[Contents: 1U blank panel x10, M6 screws x60, ...",[],38.98,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Home studio rack spacer!', 'url': ...",RackPath,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '19 x 0.39 x 1.75 inche...,B0B14TLVVZ
19998,Computers,Mediabridge™ Ethernet Cable (50 Feet) - Suppor...,4.6,402,[CAT6 / CAT5e: Supports both Cat6 and Cat5e ap...,[Mediabridge Ethernet Patch Cable Cat6 / Cat5e...,Not available,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Ethernet Cable Review', 'url': 'ht...",Mediabridge,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'Mediabridge', 'Connector Type': 'RJ...",B001W2ENX0


In [22]:
train_data.columns

Index(['main_category', 'title', 'average_rating', 'rating_number', 'features',
       'description', 'price', 'images', 'videos', 'store', 'categories',
       'details', 'parent_asin'],
      dtype='object')

In [23]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   main_category   20000 non-null  object 
 1   title           20000 non-null  object 
 2   average_rating  20000 non-null  float64
 3   rating_number   20000 non-null  int64  
 4   features        20000 non-null  object 
 5   description     20000 non-null  object 
 6   price           20000 non-null  object 
 7   images          20000 non-null  object 
 8   videos          20000 non-null  object 
 9   store           20000 non-null  object 
 10  categories      20000 non-null  object 
 11  details         20000 non-null  object 
 12  parent_asin     20000 non-null  object 
dtypes: float64(1), int64(1), object(11)
memory usage: 2.0+ MB


In [24]:
type(train_data['description'][0])

list

In [25]:
train_data['description']=train_data['description'].apply(lambda x: ' '.join(x) if isinstance(x,list) else x)

In [26]:
train_data['categories']=train_data['categories'].apply(lambda x: ' '.join(x) if isinstance(x,list) else x)

In [27]:
train_data.head(2)

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],Teleporter V3 The “Teleporter V3” kit sets a n...,Not available,[{'thumb': 'https://m.media-amazon.com/images/...,[],Fat Shark,Electronics Television & Video Video Glasses,"{'Date First Available': 'August 2, 2014', 'Ma...",B00MCW7G9M
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",HDMI In - HDMI Out,Not available,[{'thumb': 'https://m.media-amazon.com/images/...,[],SIIG,Electronics Television & Video Accessories Cab...,{'Product Dimensions': '0.83 x 4.17 x 2.05 inc...,B00YT6XQSE


In [28]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

nlp=spacy.load("en_core_web_sm")


def clean_and_extract_tags(text):
  doc=nlp(text.lower())
  tags=[token.text for token in doc if token.text.isalnum() and token.text not in STOP_WORDS]
  return ','.join(tags)

columns_to_extract_tags_from=['store','description','categories']

for column in columns_to_extract_tags_from:
  train_data[column]=train_data[column].apply(clean_and_extract_tags)

In [29]:
train_data['Tags']=train_data[columns_to_extract_tags_from].apply(lambda row: ','.join(row),axis=1)

In [30]:
train_data.head(2)

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,Tags
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],"teleporter,v3,teleporter,v3,kit,sets,new,level...",Not available,[{'thumb': 'https://m.media-amazon.com/images/...,[],"fat,shark","electronics,television,video,video,glasses","{'Date First Available': 'August 2, 2014', 'Ma...",B00MCW7G9M,"fat,shark,teleporter,v3,teleporter,v3,kit,sets..."
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]","hdmi,hdmi",Not available,[{'thumb': 'https://m.media-amazon.com/images/...,[],siig,"electronics,television,video,accessories,cable...",{'Product Dimensions': '0.83 x 4.17 x 2.05 inc...,B00YT6XQSE,"siig,hdmi,hdmi,electronics,television,video,ac..."


In [31]:
train_data['Tags'][0]

'fat,shark,teleporter,v3,teleporter,v3,kit,sets,new,level,value,fpv,world,fat,shark,renowned,performance,quality,fun,fpv,experienced,firsthand,large,screen,fpv,headset,integrated,nexwaverf,receiver,technology,simultaneously,recording,onboard,hd,footage,included,pilothd,camera,teleporter,v3,kit,comes,complete,need,step,cockpit,fpv,vehicle,included,powerful,250mw,transmitter,25,degree,fov,headset,largest,qvga,display,available,brand,new,pilothd,camera,live,av,cables,antennas,connectors,needed,electronics,television,video,video,glasses'

# **EDA**

# **Content Based Similarity based System**

In [32]:
from sklearn.feature_extraction.text import  TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [33]:
tfidf_vectorizer=TfidfVectorizer()
tfidf_matrix_content=tfidf_vectorizer.fit_transform(train_data['Tags'])
cosine_similarity_content=cosine_similarity(tfidf_matrix_content,tfidf_matrix_content)

In [44]:
train_data['title'][18000]

'Mens Flap-Over Shoulder Messenger Bag Large Capacity Vintage Canvas Laptop Bag Strong Durable with'

In [45]:
item_name='Mens Flap-Over Shoulder Messenger Bag Large Capacity Vintage Canvas Laptop Bag Strong Durable with'
item_index=train_data[train_data['title']==item_name].index[0]

In [46]:
similar_items=list(enumerate(cosine_similarity_content[item_index]))

In [48]:
similar_items=sorted(similar_items,key=lambda x: x[1],reverse=True)

In [80]:
top_similar_items=similar_items[0:11]
top_similar_items

[(18000, np.float64(1.0)),
 (5126, np.float64(0.5281254955585954)),
 (6101, np.float64(0.4727028211841911)),
 (9083, np.float64(0.47044766761776885)),
 (10747, np.float64(0.44908092719137305)),
 (4056, np.float64(0.4374674855001663)),
 (1202, np.float64(0.42393003153201114)),
 (9289, np.float64(0.41843587690614076)),
 (15803, np.float64(0.41434559001178906)),
 (7794, np.float64(0.40434246542813557)),
 (5427, np.float64(0.39969139179742985))]

In [81]:
recommended_items_indices= [x[0] for x in top_similar_items]
recommended_items_indices

[18000, 5126, 6101, 9083, 10747, 4056, 1202, 9289, 15803, 7794, 5427]

In [82]:
pd.set_option('display.max_colwidth', None)

In [83]:
train_data.iloc[recommended_items_indices][['title','store']]

,title,store
18000,Mens Flap-Over Shoulder Messenger Bag Large Capacity Vintage Canvas Laptop Bag Strong Durable with,
5126,Handmadecraft Leather Unisex Real Leather Messenger Bag for Laptop Briefcase Satchel,handolederco
6101,Baosha 17 inch Canvas Laptop Computer Bag Messenger Bag Briefcase Large Satchel Shoulder Bag BC-12 (Black),baosha
9083,Laptop Bag Motorcycle Laptop Case Computer Case 13-15.6 inch Laptop Sleeve,mnsruu
10747,"Laptop Tote Bag for Women, 13.3-14 inch Waterproof PU Leather Work Briefcase with USB Charging Port Computer Shoulder Bag, Office Messenger Business Handbag Gifts for Women Work Travel, Black",taygeer
4056,"Targus CityLite Laptop Briefcase Shoulder Messenger Bag for 15.6-Inch Laptop, Black (TBT053US)",targus
1202,Charm&Magic Large Waterproof Canvas SLR/DSLR Digital Camera Messager Bag Laptop Causual Shoulder Bag with Shockproof Insert,"charm,magic"
9289,"KAKA 17 Inch Laptop Travel Backpack Casual Daypack with Side Handle and Tie Rod Fixing Belt，Anti Theft Water Resistant Student Back Pack,Commuting Backpack for Men Women",kaka
15803,Canvaslife Big White Rose Patten Waterproof Laptop Shoulder Messenger Bag Case Sleeve for 14 Inch 15 Inch Laptop MacBook Pro 15 Case Laptop Briefcase 15.6 Inch,canvaslife
7794,DACHEE Big White Rose Patten Waterproof Laptop Shoulder Messenger Bag Case Sleeve for 11 Inch 12 Inch 13 Inch Laptop,dachee
